# 04 - Evaluación y comparación de modelos de regresión

En este notebook se resumen y visualizan los resultados de los modelos supervisados de regresión
entrenados para predecir la variable continua `TotalPay`.

Se comparan los modelos mediante las métricas:
- **MAE** (Mean Absolute Error)
- **RMSE** (Root Mean Squared Error)
- **R²** (coeficiente de determinación)

y se genera una figura para incluir en el `README.md`.

In [1]:
# Imports básicos y carpeta de figuras
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Ruta coherente con el resto del proyecto
RUTA_FIGURAS = Path("../reports/figures")
RUTA_FIGURAS.mkdir(parents=True, exist_ok=True)

print(f"Carpeta lista: {RUTA_FIGURAS.resolve()}")

Carpeta lista: G:\ProyectosPython\proyecto1\desarrollo\reports\figures


In [2]:
# Resultados de modelos de regresión (copiados de los resultados del modelado)

resultados_reg = [
    {"Modelo": "Regresión Lineal",              "MAE": 21797.6081, "RMSE": 30708.8907, "R": 0.6177},
    {"Modelo": "Ridge alpha=10",               "MAE": 21797.1362, "RMSE": 30708.8663, "R": 0.6177},
    {"Modelo": "Decision Tree Regressor d=8",  "MAE": 11062.6442, "RMSE": 22686.6121, "R": 0.7914},
    {"Modelo": "Random Forest Regressor",      "MAE":  8896.1063, "RMSE": 21747.0436, "R": 0.8083},
]

dfreg = pd.DataFrame(resultados_reg).set_index("Modelo")

print("Comparación de Modelos de Regresión (Test)")
print(dfreg.round(4).to_string())
print()
print(f"Mejor R:     {dfreg['R'].idxmax()}")
print(f"Menor RMSE:  {dfreg['RMSE'].idxmin()}")

Comparación de Modelos de Regresión (Test)
                                    MAE        RMSE       R
Modelo                                                     
Regresión Lineal             21797.6081  30708.8907  0.6177
Ridge alpha=10               21797.1362  30708.8663  0.6177
Decision Tree Regressor d=8  11062.6442  22686.6121  0.7914
Random Forest Regressor       8896.1063  21747.0436  0.8083

Mejor R:     Random Forest Regressor
Menor RMSE:  Random Forest Regressor


In [3]:
# Tabla de modelos de regresión como imagen (alta resolución)

import matplotlib.pyplot as plt
from pathlib import Path

RUTA_FIGURAS = Path("../reports/figures")
RUTA_FIGURAS.mkdir(parents=True, exist_ok=True)

# Copia de la tabla y redondeo
df_tabla_reg = dfreg.copy().round(2)

fig, ax = plt.subplots(figsize=(12, 2.8))
ax.axis("off")

tabla = ax.table(
    cellText=df_tabla_reg.values,
    colLabels=df_tabla_reg.columns,
    rowLabels=df_tabla_reg.index,
    loc="center",
    cellLoc="center"
)

tabla.auto_set_font_size(False)
tabla.set_fontsize(10)
tabla.scale(1.1, 1.6)

# Estilo encabezado columnas
for (row, col), cell in tabla.get_celld().items():
    if row == 0:
        # Esquina superior izquierda (celda vacío de rowLabels)
        cell.set_text_props(weight="bold", color="white")
        cell.set_facecolor("#1F4E79")
    elif row == 0 and col > 0:
        cell.set_text_props(weight="bold", color="white")
        cell.set_facecolor("#1F4E79")

# Estilo encabezado filas (nombres de modelos)
for i in range(1, len(df_tabla_reg.index) + 1):
    celda_fila = tabla[(i, 0)]
    celda_fila.set_text_props(weight="bold")

output_tabla_reg = RUTA_FIGURAS / "tabla_modelos_regresion.png"
plt.tight_layout()
plt.savefig(output_tabla_reg, dpi=500, bbox_inches="tight")
plt.close()

print(f"Tabla de regresión guardada en: {output_tabla_reg}")

Tabla de regresión guardada en: ..\reports\figures\tabla_modelos_regresion.png


In [4]:
# Comparación visual de modelos de regresión (alta resolución con valores)

import matplotlib.pyplot as plt
import os
from pathlib import Path

RUTA_FIGURAS = Path("../reports/figures")
RUTA_FIGURAS.mkdir(parents=True, exist_ok=True)

# Copiamos y ordenamos por R descendente
df_plot = dfreg.copy().sort_values("R", ascending=False)

modelos = df_plot.index.tolist()
mae = df_plot["MAE"].values
rmse = df_plot["RMSE"].values
r2 = df_plot["R"].values

x = range(len(modelos))
width = 0.25

plt.style.use("seaborn-v0_8-whitegrid")
fig, ax1 = plt.subplots(figsize=(12, 7))  # más grande para legibilidad

# Barras de MAE y RMSE (eje izquierdo)
bars_mae = ax1.bar(
    [i - width/2 for i in x],
    mae,
    width=width,
    label="MAE",
    color="#4C72B0",
)
bars_rmse = ax1.bar(
    [i + width/2 for i in x],
    rmse,
    width=width,
    label="RMSE",
    color="#55A868",
)

ax1.set_ylabel("Error (USD)")
ax1.tick_params(axis="y")

# Eje derecho para R²
ax2 = ax1.twinx()
ax2.plot(x, r2, marker="o", color="#C44E52", linewidth=2, label="R²")
ax2.set_ylabel("R²")
ax2.set_ylim(0, 1.0)

# Etiquetas del eje x
ax1.set_xticks(list(x))
ax1.set_xticklabels(modelos, rotation=15, ha="right")

ax1.set_title("Comparación de modelos de regresión (conjunto de prueba)")

# --- Anotar valores en las barras (MAE y RMSE) ---
def anotar_barras(barras, valores):
    for bar, val in zip(barras, valores):
        altura = bar.get_height()
        ax1.text(
            bar.get_x() + bar.get_width()/2,
            altura,
            f"{val:,.0f}",            # sin decimales, con separador de miles
            ha="center",
            va="bottom",
            fontsize=9,
            rotation=90,              # vertical para no chocar
        )

anotar_barras(bars_mae, mae)
anotar_barras(bars_rmse, rmse)

# --- Anotar valores de R² sobre los puntos ---
for i, val in enumerate(r2):
    ax2.text(
        i,
        val + 0.02,                  # un poquito arriba del punto
        f"{val:.2f}",
        ha="center",
        va="bottom",
        fontsize=9,
        color="#C44E52",
    )

# Leyenda combinada
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc="upper right")

plt.tight_layout()

output_path = RUTA_FIGURAS / "modelos_regresion_comparacion.png"
plt.savefig(output_path, dpi=400, bbox_inches="tight")  # alta resolución
plt.close()

print(f"Figura guardada en alta resolución en: {output_path}")

Figura guardada en alta resolución en: ..\reports\figures\modelos_regresion_comparacion.png


### Conclusiones sobre los modelos de regresión

- Los modelos lineales (`Regresión Lineal` y `Ridge alpha=10`) sirven como línea base y explican ~62 % de la varianza (R² ≈ 0.62), con errores MAE y RMSE relativamente altos.
- `Decision Tree Regressor d=8` mejora notablemente el MAE y RMSE, elevando el R² a ~0.79.
- `Random Forest Regressor` es el mejor modelo global: presenta el **menor MAE y RMSE**, y el **mayor R²** (~0.81), por lo que se selecciona como modelo final para predecir `TotalPay`.